# Análisis de ventas retail y devoluciones

Este cuaderno analiza el rendimiento real de cada canal de venta, teniendo en cuenta la tasa de devolución y el ingreso neto real.

## Objetivo
- Evaluar qué canales generan valor neto real.
- Detectar categorías con mayor tasa de devolución.
- Convertir los datos en decisiones operativas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Carga de datos
try:
    df_pedidos = pd.read_csv('pedidos.csv')
    df_productos = pd.read_csv('productos.csv')
    df_campanas = pd.read_csv('campana.csv')
    df_lineas = pd.read_csv('lineas_pedido.csv')
    df_devoluciones = pd.read_csv('devoluciones.csv')
    df_clientes = pd.read_csv('clientes.csv')
    print('Archivos cargados correctamente')
except FileNotFoundError as e:
    print(f'Archivo no encontrado: {e}')

In [ ]:
# Limpieza de categorías y normalización de datos
reemplazos = {
    'vestido': 'vestidos',
    'camiseta': 'camisetas',
    'pantalon': 'pantalones',
    'falda': 'faldas',
    'abrigo': 'abrigos',
    'zapatos': 'calzado',
    'accesorio': 'accesorios',
    'complementos': 'accesorios',
    'jersey': 'jerseis',
    'jerséis': 'jerseis'
}

df_productos['categoria_limpia'] = df_productos['categoria'].str.strip().str.lower().replace(reemplazos)

df_lineas['precio_unitario'] = df_lineas['precio_unitario']
    .astype(str)
    .str.replace('€', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)

df_lineas['sub_total_sin_descuento'] = df_lineas['cantidad'] * df_lineas['precio_unitario']
df_lineas['descuento_pct'] = pd.to_numeric(
    df_lineas['descuento_pct'].astype(str).str.replace(',', '.', regex=False).str.replace(r'[^\d.]', '', regex=True),
    errors='coerce'
)
df_lineas.loc[df_lineas['descuento_pct'] >= 1, 'descuento_pct'] = df_lineas['descuento_pct'] / 100
df_lineas['total_linea'] = df_lineas['sub_total_sin_descuento'] * (1 - df_lineas['descuento_pct'])

df_completo = pd.merge(df_lineas, df_productos, on='producto_id', how='inner')
df_con_devoluciones = pd.merge(df_completo, df_devoluciones, on='linea_id', how='left')
df_con_devoluciones['fue_devuelto'] = df_con_devoluciones['devolucion_id'].notna()
df_pedidos['fecha_pedido'] = pd.to_datetime(df_pedidos['fecha_pedido'], dayfirst=True, format='mixed')
df_final = pd.merge(df_con_devoluciones, df_pedidos, on='pedido_id', how='left')
print('Limpieza y enriquecimiento completados')

In [ ]:
# Resumen financiero por canal
df_final['importe_devuelto'] = np.where(df_final['fue_devuelto'], df_final['total_linea'], 0.0)
df_resumen_canales = df_final.groupby('canal').agg(
    ventas_brutas=('total_linea', 'sum'),
    total_devoluciones_eur=('importe_devuelto', 'sum'),
    tasa_devolucion_pct=('fue_devuelto', lambda x: x.mean() * 100)
).reset_index()

df_resumen_canales['ingreso_neto_real'] = df_resumen_canales['ventas_brutas'] - df_resumen_canales['total_devoluciones_eur']
df_resumen_canales = df_resumen_canales.sort_values('ingreso_neto_real', ascending=False)
df_resumen_canales.to_csv('reporte_canales_2024.csv', index=False)
display(df_resumen_canales)

## Hallazgos principales

- Instagram aparece como el canal con mejor rendimiento neto.
- La tasa de devolución penaliza más a las categorías con problemas de ajuste y talla.
- El negocio debe mirar el ingreso neto real y no solo las ventas brutas.

In [ ]:
# Visualización de rendimiento por canal
sns.set_theme(style='whitegrid')
df_melted = df_resumen_canales.melt(
    id_vars='canal',
    value_vars=['ventas_brutas', 'ingreso_neto_real'],
    var_name='tipo_ingreso',
    value_name='importe'
)

df_melted['tipo_ingreso'] = df_melted['tipo_ingreso'].map({
    'ventas_brutas': 'Ventas brutas',
    'ingreso_neto_real': 'Ingreso neto real'
})

plt.figure(figsize=(12, 5))
sns.barplot(data=df_melted, x='canal', y='importe', hue='tipo_ingreso', palette=['#34495e', '#2ecc71'])
plt.title('Ventas brutas vs. ingreso neto real por canal')
plt.xlabel('Canal')
plt.ylabel('Euros (€)')
plt.tight_layout()
plt.show()